# Text Preprocessing: Tokenization to Lemmatization

Day 1, Session 2. Sentence segmentation, tokenization, normalization, stopwords, stemming and lemmatization

Run this in Google Colab (or any Jupyter setup). Run the cells top to bottom.

## Setup

In [ ]:
# one time installs, colab already has most of these
# nltk  -> sentence splitter, tokenizer, stopwords, stemmer, wordnet lemmatizer, movie review corpus
# spacy -> second opinion on tokens and lemmas
%pip install -q nltk spacy
!python -m spacy download en_core_web_sm

In [ ]:
import re
import nltk
import spacy

# data packages nltk needs for this session
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4", "movie_reviews", "brown"]:
    nltk.download(pkg, quiet=True)

from nltk.corpus import movie_reviews, stopwords

nlp = spacy.load("en_core_web_sm")

## 1. Sentence segmentation

First the naive way, then the trained splitter.

In [ ]:
text = "Dr. Rao paid 3.5 lakh. Then he left."
print(text.split("."))
print(nltk.sent_tokenize(text))

In [ ]:
# spacy decides sentence boundaries from the parse
doc = nlp("Dr. Rao paid 3.5 lakh. Then he left.")
print([s.text for s in doc.sents])

## 2. Tokenization

In [ ]:
text = "I can't wait for 5:30pm, e-mail me!"
print(text.split())
print(nltk.word_tokenize(text))
print([t.text for t in nlp(text)])

In [ ]:
# tweet style text is where tokenizers differ most
tweet = "loved it!!! #NLP @rao_sir http://t.co/abc :) gr8"
print(nltk.word_tokenize(tweet))
print([t.text for t in nlp(tweet)])

# nltk also has a tweet tokenizer, it keeps hashtags and mentions together
from nltk.tokenize import TweetTokenizer
print(TweetTokenizer().tokenize(tweet))

## 3. Normalization

In [ ]:
raw = "  Visit   NLP Lab!!  It's Cool... 100% sure :) "

clean = re.sub(r"[^a-z0-9\s]", "", raw.lower())   # keep only letters, digits, spaces
clean = re.sub(r"\s+", " ", clean).strip()          # squeeze spaces
print(clean)

In [ ]:
# the cost of lowercasing: US and us become the same token
for s in ["US troops left", "Give us time"]:
    print(s.lower().split())

In [ ]:
# true-casing: learn the usual casing of each word from a corpus
# (movie_reviews is all lowercase, so we use brown news text here)
from collections import Counter
from nltk.corpus import brown

forms = Counter(brown.words(categories="news"))
usual = {}
for w, c in forms.items():
    lw = w.lower()
    if lw not in usual or c > forms[usual[lw]]:
        usual[lw] = w

def truecase(sentence):
    return " ".join(usual.get(w.lower(), w) for w in sentence.split())

# plain lowercasing would lose Texas and American, this restores them
# it is not perfect: President wins over president because news text uses it more
print(truecase("THE president visited TEXAS and the AMERICAN senate"))

## 4. Stopword removal

In [ ]:
from nltk.corpus import stopwords

stop = set(stopwords.words("english"))
print(len(stop), sorted(stop)[:6])

words = "this movie is not good at all".split()
print([w for w in words if w not in stop])

In [ ]:
# fix: take the negation words out of the list
stop_keep_neg = stop - {"not", "no", "nor"}
print([w for w in words if w not in stop_keep_neg])

## 5. Stemming and lemmatization

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

ps, wl = PorterStemmer(), WordNetLemmatizer()

for w, pos in [("studies", "n"), ("studying", "v"),
               ("better", "a"), ("mice", "n")]:
    print(w, ps.stem(w), wl.lemmatize(w, pos))

In [ ]:
# without the pos tag the lemmatizer assumes noun
print(wl.lemmatize("studying"))
print(wl.lemmatize("studying", pos="v"))

# stemmers can merge unrelated words
print(ps.stem("university"), ps.stem("universe"))

In [ ]:
# spacy gives lemmas using its own tagger, no need to pass pos
# the small model is not perfect: mice stays mice and better becomes well
doc = nlp("The studies were showing better results than the running mice experiments.")
print([(t.text, t.lemma_) for t in doc])

## 6. Effect on vocabulary size

How much does each step shrink the vocabulary on real reviews?

In [ ]:
from nltk.corpus import movie_reviews

fileids = movie_reviews.fileids()
words = movie_reviews.words(fileids[:200])
low = [w.lower() for w in words if w.isalpha()]
nostop = [w for w in low if w not in stop]
stems = [ps.stem(w) for w in nostop]
print("raw      ", len(set(words)))
print("lowercase", len(set(low)))
print("no stops ", len(set(nostop)))
print("stemmed  ", len(set(stems)))

## 7. One reusable preprocessing function

In [ ]:
def preprocess(text, keep_negation=True, mode="lemma"):
    # mode can be 'lemma', 'stem' or 'none'
    stops = set(stopwords.words("english"))
    if keep_negation:
        stops -= {"not", "no", "nor"}
    out = []
    for t in nlp(text.lower()):
        if not t.is_alpha or t.text in stops:
            continue
        if mode == "lemma":
            out.append(t.lemma_)
        elif mode == "stem":
            out.append(ps.stem(t.text))
        else:
            out.append(t.text)
    return out

sample = "I did not enjoy the movies, the acting was getting worse."
print(preprocess(sample, mode="lemma"))
print(preprocess(sample, mode="stem"))
print(preprocess(sample, keep_negation=False))

## Practice

1. (Easy) Tokenize 'Prices rose by 3.5% on Monday, said Dr. Mehta.' with split() and with word_tokenize. List every difference.

2. (Medium) Write preprocess(text, keep_negation=True) that lowercases, tokenizes, removes stopwords but keeps not, no and nor when keep_negation is True.

3. (Challenge) On 500 movie reviews, compare vocabulary size after stemming with the vocabulary size after spaCy lemmatization. Which is smaller and why?

In [ ]:
# your practice code here